# Outcome analysis

Analyze saved original-prompt responses for housing differences, correlations, directional recovery, and CNEP+ comparisons

Run from the project root or `analysis/`


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((candidate for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent)
             if (candidate / "analysis" / "analysis_utils.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the project root or its analysis folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Resolve inputs from the project root
os.chdir(ROOT)

import pandas as pd
import numpy as np
from analysis.analysis_utils import spearmanr
from itertools import combinations

from analysis.analysis_utils import DATA_DIR, RESULT_DIR, FIGURE_DIR, normalize_knowledge, score_literature, paired_cnep_mad, paired_housing_effect, reconstruct_original_profiles, PAPER_PAIRS, COLUMNS
RESULT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
B = 1000
SEED = 42
EXPECTED_PROFILES = 224

def require_unique(frame, keys):
    if frame[keys].isna().any().any() or frame.duplicated(keys).any():
        raise ValueError(f"Missing or duplicate identifiers: {keys}. Inspect repeated runs before analysis.")

def rho(x, y):
    a = np.column_stack([np.asarray(x, dtype=float), np.asarray(y, dtype=float)])
    a = a[np.isfinite(a).all(axis=1)]
    if len(a) < 3 or np.unique(a[:, 0]).size < 2 or np.unique(a[:, 1]).size < 2:
        return np.nan
    return float(spearmanr(a[:, 0], a[:, 1]).statistic)


In [ ]:
knowledge_df = normalize_knowledge(pd.read_csv(DATA_DIR / "knowledge_outputs.csv"))
# Normalize saved knowledge responses
print("Loaded the 32 retained knowledge relationships for the original-prompt literature comparison.")


In [ ]:
file = '2018-09-13--06.02.24.events.csv'
cnep_raw = pd.read_csv(file, header=4)
df = pd.read_csv(DATA_DIR / "outputs.csv")
require_unique(df, ["Model", "Agent", "Paraphrase", "Homeless"])
if not df["Homeless"].isin([0, 1]).all() or not df["Paraphrase"].isin([0, 1, 2]).all():
    raise ValueError("Invalid condition labels.")
# Use the original behavioral prompt
df = df.loc[df.Paraphrase.eq(0)].copy()
hcv = df["HCV"].astype("string").str.strip().str.lower()
hcv_map = {"susceptible":0, "acute":1}
if not hcv.dropna().isin(hcv_map).all():
    raise ValueError("Unrecognized HCV state; inspect before coding active infection.")
df["HCV_active"] = hcv.map(hcv_map).astype(float)
measures = ["Overdose", "Public_prop", "Recept_sharing", "Partners", "Drug_in", "Drug_out",
            "Partner_HCV_prop", "Treatment", "Injections", "Alt_route", "HCV_active"]
for col in measures:
    df[col] = pd.to_numeric(df[col], errors="raise")
    if np.isinf(df[col]).any() or (df[col].dropna() < 0).any():
        raise ValueError(f"Invalid numeric outcome: {col}")
for col in ["Public_prop", "Recept_sharing", "Partner_HCV_prop", "Treatment", "HCV_active"]:
    if (df[col].dropna() > 1).any(): raise ValueError(f"Out-of-range proportion: {col}")
if not df["Treatment"].dropna().isin([0,1]).all(): raise ValueError("Treatment must be binary.")
df_f = df.loc[df.Paraphrase.eq(0)].copy()
assert set(df.Model) == {"claudesonnet4.6", "deepseek.v3.2", "gpt-oss-120b"}


In [ ]:
# Reconstruct the original profiles and strata
mad_profiles = reconstruct_original_profiles(ROOT / file, set(df_f.Agent))
sampled_agents = mad_profiles.Agent.to_numpy()
if len(sampled_agents) != EXPECTED_PROFILES or len(set(sampled_agents)) != EXPECTED_PROFILES:
    raise ValueError("Expected 224 distinct CNEP+ profiles.")
for model, group in df_f.groupby("Model"):
    for housing in [0, 1]:
        observed = set(group.loc[group.Homeless.eq(housing), "Agent"])
        if observed != set(sampled_agents):
            raise ValueError(f"{model}, housing={housing}: output IDs differ from the original 224 profiles.")


## CNEP+ comparison


In [ ]:
# Time is measured in years
FOLLOWUP_INTERVAL = 0.5
cnep = cnep_raw.loc[cnep_raw.Agent.isin(sampled_agents)].copy()
cnep["Time"] = pd.to_numeric(cnep["Time"], errors="raise")
cnep["HCV"] = cnep["HCV"].astype("string").str.strip().str.lower().replace({"infectiousacute":"acute"})
# Baseline is the earliest activation
activation = (cnep.loc[cnep.Event.eq("activated")].sort_values(["Agent", "Time"], kind="stable")
              .drop_duplicates("Agent", keep="first"))
if set(activation.Agent) != set(sampled_agents): raise ValueError("Missing activation rows for paper cohort.")
first_times = activation[["Agent", "Time"]].rename(columns={"Time":"first_time"})
VALID_HCV = ["acute", "chronic", "susceptible", "exposed", "recovered"]
# Normalize HCV labels and exclude vaccinated records
cnep = cnep.loc[cnep.HCV.isin(VALID_HCV)].merge(first_times, on="Agent", validate="many_to_one")
cnep["target_time"] = cnep.first_time + FOLLOWUP_INTERVAL
cnep_followup = cnep.loc[cnep.Time.gt(cnep.first_time)].copy()
cnep_followup["time_diff"] = (cnep_followup.Time - cnep_followup.target_time).abs()
snapshot = (cnep_followup.sort_values(["Agent", "time_diff", "Time"], kind="stable")
            .drop_duplicates("Agent", keep="first").rename(columns={"Time":"Time_used"}))
if set(snapshot.Agent) != set(sampled_agents): raise ValueError("Missing follow-up records for paper cohort.")
snapshot["actual_diff"] = snapshot.Time_used - snapshot.first_time
snapshot[["Agent", "first_time", "Time_used", "actual_diff", "time_diff"]].to_csv(
    RESULT_DIR / "cnep_followup_audit.csv", index=False)
print(snapshot[["actual_diff", "time_diff"]].describe())
cnep_agg = snapshot.copy()
for dest, source in {"Recept_sharing":"Fraction_recept_sharing", "Drug_in":"Drug_in_degree",
                     "Drug_out":"Drug_out_degree", "Partner_HCV_prop":"HCV_friend_preval"}.items():
    cnep_agg[dest] = cnep_agg[source]
cnep_agg["Injections"] = cnep_agg.Daily_injection_intensity * 7
cnep_agg["HCV_active"] = cnep_agg.HCV.isin(["acute", "chronic"]).astype(int)

In [ ]:
compare_cols = ["Injections", "Recept_sharing", "Drug_in",
    "Drug_out", "Partner_HCV_prop"]
tolerances = [10, 5, 2, 1, 0.5, 0.25, 0.1]
MIN_AGENTS = 10

require_unique(snapshot, ["Agent"])
require_unique(df_f, ["Model", "Agent", "Homeless"])

# Use the selected CNEP+ follow-up records
cnep_reference = snapshot[[
        "Agent", "actual_diff", "Daily_injection_intensity",
        "Fraction_recept_sharing", "Drug_in_degree",
        "Drug_out_degree", "HCV_friend_preval"]].copy()

cnep_reference["target_error"] = (cnep_reference["actual_diff"] - 0.5).abs()

cnep_reference["Injections"] = (cnep_reference["Daily_injection_intensity"] * 7)
cnep_reference["Recept_sharing"] = (cnep_reference["Fraction_recept_sharing"])
cnep_reference["Drug_in"] = cnep_reference["Drug_in_degree"]
cnep_reference["Drug_out"] = cnep_reference["Drug_out_degree"]
cnep_reference["Partner_HCV_prop"] = cnep_reference["HCV_friend_preval"]

cnep_reference = cnep_reference[["Agent", "target_error"] + compare_cols
].rename(columns={c: f"CNEP_{c}" for c in compare_cols})


def correlation_mad(frame):
    """Compare LLM and CNEP+ correlations using both housing conditions."""
    differences = []

    for var_a, var_b in combinations(compare_cols, 2):
        columns = [var_a, var_b, f"CNEP_{var_a}", f"CNEP_{var_b}"]
        values = frame[columns].to_numpy(dtype=float)
        values = values[np.isfinite(values).all(axis=1)]

        llm_rho = rho(values[:, 0], values[:, 1])
        cnep_rho = rho(values[:, 2], values[:, 3])

        if np.isfinite(llm_rho) and np.isfinite(cnep_rho):
            differences.append(abs(llm_rho - cnep_rho))

    return (float(np.mean(differences)) if differences else np.nan,
        len(differences))

sweep_rows = []

for model, model_rows in df_f.groupby("Model", sort=True):
    model_rows = model_rows.copy()

    # Require both housing conditions for each profile
    condition_sets = model_rows.groupby("Agent")["Homeless"].agg(
        lambda values: set(values))
    if not condition_sets.map(lambda values: values == {0, 1}).all():
        raise ValueError(
            f"{model}: each agent must have both housing conditions.")

    merged = model_rows.merge(
        cnep_reference,
        on="Agent",
        how="left",
        validate="many_to_one",
        indicator=True)
    if not merged["_merge"].eq("both").all():
        raise ValueError(f"{model}: some agents lack an CNEP+ snapshot.")
    merged = merged.drop(columns="_merge")

    for tolerance in tolerances:
        retained = merged[merged["target_error"] <= tolerance].copy()
        n_agents = retained["Agent"].nunique()

        if n_agents >= MIN_AGENTS:
            mad, n_pairs = correlation_mad(retained)
            status = "ok" if n_pairs else "no estimable pairs"
        else:
            mad, n_pairs = np.nan, 0
            status = f"fewer than {MIN_AGENTS} agents"

        sweep_rows.append({
            "Model": model,
            "tolerance_years": tolerance,
            "n_agents": n_agents,
            "n_pairs": n_pairs,
            "MAD": mad,
            "status": status})

sweep_df = pd.DataFrame(sweep_rows).sort_values(
    ["Model", "tolerance_years"], ascending=[True, False]).reset_index(drop=True)

Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)
sweep_df.to_csv(
    Path(RESULT_DIR) / "cnep_mad_tolerance_sweep.csv",
    index=False)
print(sweep_df.to_string(index=False))

from analysis.artwork import plot_timing_sensitivity
plot_timing_sensitivity(sweep_df)


In [ ]:
compare_cols = ["Injections", "Recept_sharing", "Drug_in", "Drug_out", "Partner_HCV_prop"]
compare_pairs = list(combinations(compare_cols, 2))
cnep_agg_renamed = cnep_agg[["Agent"] + compare_cols].rename(columns={c: "CNEP_" + c for c in compare_cols})

def cnep_pair_vectors(frame):
    llm_values, cnep_values = [], []
    for x, y in compare_pairs:
        # Match complete observations across both sources
        a = frame[[x, y, "CNEP_" + x, "CNEP_" + y]].replace([np.inf, -np.inf], np.nan).dropna()
        llm_values.append(rho(a[x], a[y]))
        cnep_values.append(rho(a["CNEP_" + x], a["CNEP_" + y]))
    return np.asarray(llm_values), np.asarray(cnep_values)

cnep_results = []

for (model, housing), sub in df_f.groupby(["Model", "Homeless"]):
    matched = sub.merge(
        cnep_agg_renamed,
        on="Agent",
        how="left",
        validate="one_to_one",
        indicator=True
    )

    if not matched["_merge"].eq("both").all():
        raise ValueError("Unmatched CNEP+ IDs.")

    lv, av = cnep_pair_vectors(matched)

    eligible = np.isfinite(lv) & np.isfinite(av)

    estimate = (
        np.abs(lv[eligible] - av[eligible]).mean()
        if eligible.any()
        else np.nan
    )

    cnep_results.append(
        dict(
            Model=model,
            Housing=housing,
            N_agents=len(matched),
            N_pairs=int(eligible.sum()),
            N_pairs_prespecified=len(compare_pairs),
            mean_corr_difference=estimate,
            **{
                f"assoc_LLM_{x}__{y}": v
                for (x, y), v in zip(compare_pairs, lv)
            },
            **{
                f"assoc_CNEP_{x}__{y}": v
                for (x, y), v in zip(compare_pairs, av)
            }
        )
    )

cnep_results_df = pd.DataFrame(cnep_results)
results_df = cnep_results_df

cnep_results_df.to_csv(RESULT_DIR / "cnep_mad.csv", index=False)

print(
    cnep_results_df[
        [
            "Model",
            "Housing",
            "N_agents",
            "N_pairs",
            "mean_corr_difference"
        ]
    ]
)


In [ ]:
from analysis.artwork import plot_cnep_correlations
plot_cnep_correlations(results_df)


In [ ]:
# Resample within the original profile strata
# Keep both housing rows and the matched CNEP+ record together
pooled_cnep_results_df = pd.DataFrame([
    {"Model": model, **paired_cnep_mad(group.merge(cnep_agg_renamed, on="Agent", validate="many_to_one"), compare_cols, B, SEED, profiles=mad_profiles)}
    for model, group in df_f.groupby("Model")
])
pooled_cnep_results_df.to_csv(RESULT_DIR / "cnep_mad_pooled.csv", index=False)
print(pooled_cnep_results_df.round(3).to_string(index=False))


## Housing differences

Calculate mean paired differences: homeless minus stable

Counts use weekly units; proportion and binary outcomes use percentage points

Housing differences have no confidence intervals

Run `bootstrap.ipynb` for directional agreement and estimability


In [ ]:
housing_results = pd.DataFrame([
    {"Model":model, "Variable":v, "Paraphrase":0, **paired_housing_effect(group, v)}
    for model, group in df_f.groupby("Model") for v in measures])
housing_results.to_csv(RESULT_DIR / "housing_effects.csv", index=False)
print(housing_results.to_string(index=False))

## Pooled correlations


In [ ]:
# Calculate the 23 prespecified nonhousing correlations
allowed_pairs = {frozenset((COLUMNS[x], COLUMNS[y])) for x, y in PAPER_PAIRS
                 if COLUMNS[x] != "Homeless"}

correlation_pairs = [pair for pair in combinations(measures, 2)
                     if frozenset(pair) in allowed_pairs]
if len(correlation_pairs) != 23:
    raise ValueError("Expected 23 distinct prespecified nonhousing relationships.")
all_results = []

for model_name, df_model in df_f.groupby("Model"):
    results = []

    for a, b in correlation_pairs:
        df_xy = pd.concat([df_model[a], df_model[b]], axis=1).dropna()
        x = df_xy.iloc[:, 0]
        y = df_xy.iloc[:, 1]
        if x.nunique() < 2 or y.nunique() < 2:
            corr = np.nan
        else:
            corr = rho(x, y)

        results.append({"model": model_name,
            "var1": a,
            "var2": b,
            "correlation": corr})

    all_results.append(pd.DataFrame(results))

corr_df = pd.concat(all_results, ignore_index=True)
# Pool both housing conditions
corr_df["scope"] = "pooled housing conditions"
corr_df.to_csv(RESULT_DIR / "pooled_outcome_correlations.csv", index=False)

## Literature scoring

Score the 28 directional targets; leave the four mixed-expectation targets unscored

Run `prompt_sensitivity.ipynb` for comparisons between prompt versions


In [ ]:
literature_audit, literature_summary = score_literature(housing_results, corr_df, knowledge_df)
literature_audit.to_csv(RESULT_DIR / "literature_target_audit.csv", index=False)
literature_summary.to_csv(RESULT_DIR / "literature_summary.csv", index=False)
print(literature_summary.round(3).to_string(index=False))
